In [ ]:
# IMport & Data Loading 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_recall_curve, auc, f1_score, 
                             confusion_matrix, classification_report, PrecisionRecallDisplay)
from sklearn.model_selection import StratifiedKFold, cross_val_score

# 1. Load the E-commerce Processed Data
X_train = pd.read_csv('../data/processed/X_train_final.csv')
y_train = pd.read_csv('../data/processed/y_train_final.csv').values.ravel()
X_test = pd.read_csv('../data/processed/X_test_final.csv

In [ ]:
# Create a Universal Evaluation Function
def evaluate_model(model, X_test, y_test, model_name="Model"):
    # Predictions
    y_pred = model.predict(X_test)
    y_probs = model.predict_proba(X_test)[:, 1]
    
    # Calculate Metrics
    precision, recall, _ = precision_recall_curve(y_test, y_probs)
    auc_pr = auc(recall, precision)
    f1 = f1_score(y_test, y_pred)
    
    print(f"--- {model_name} Evaluation ---")
    print(f"AUC-PR: {auc_pr:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print("\nConfusion Matrix:")
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
    plt.show()
    
    return {"Model": model_name, "AUC-PR": auc_pr, "F1-Score": f1}

In [ ]:
# Build Baseline (Logistic Regression)
# Initialize and train
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Evaluate
lr_results = evaluate_model(lr_model, X_test, y_test, "Logistic Regression (Baseline)")

In [ ]:
# Build Ensemble Model (Random Forest)
# Initialize Random Forest with tuned parameters
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate
rf_results = evaluate_model(rf_model, X_test, y_test, "Random Forest (Ensemble)")

In [ ]:
# Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Perform CV for Random Forest
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=skf, scoring='f1')

print(f"Random Forest F1-Score CV: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

In [ ]:
# Final Comparison Table & Selection Justification
# Create a comparison table
comparison_df = pd.DataFrame([lr_results, rf_results])
print(comparison_df)

In [ ]:
# Save Your Models
import joblib

joblib.dump(lr_model, '../models/logistic_regression_model.pkl')
joblib.dump(rf_model, '../models/random_forest_model.pkl')

print("Models saved in models/ directory.")